In [ ]:
# Setup — all imports live here so the notebook executes top-to-bottom
# without reimporting mid-notebook. When running headlessly (CI, HPC),
# uncomment the ``matplotlib.use('Agg')`` line *before* the pyplot import
# so figures never need an interactive display.
import dataclasses
import logging
import os
import sys
from pathlib import Path

# import matplotlib
# matplotlib.use('Agg')  # enable for headless runs
import matplotlib.pyplot as plt
import mne
import numpy as np
import seaborn as sns
from mne.viz import plot_topomap
from scipy.signal import spectrogram as sp_spectrogram

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from sklearn.decomposition import PCA, FastICA  # noqa: E402

from scripts.analysis_common import (  # noqa: E402
    FREQUENCY_BANDS,
    WAVELET_BAND_FREQ_RESOLUTION_HZ,
    analyzers_to_datasets,
    load_analyzers,
    wavelet_transform,
)
from src.analysis.wavelet_ica import zscore_by_time  # noqa: E402
from src.definitions.constants import ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    ExclusionCategories,
    MusicTypeVariants,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
%matplotlib inline

# Inverted Super-Brain ICA / PCA on Wavelet Power

## Approach overview

**Approach 4 — "Inverted super-brain" decomposition.**  We use the same
2-D matrix as the super-brain approach but **swap observations and features**:
*person × channel × frequency* becomes the observation (sample) axis and
*time* becomes the feature axis:

```
Input:   (n_subjects, n_channels, n_freqs, n_times)  — 4-D wavelet power
Reshape: (n_subjects × n_channels × n_freqs,  n_times)
         ──────── observations ──────────────  features
```

The resulting 2-D matrix has **S × C × F rows** (one per subject–electrode–frequency
combination) and **T columns** (time points).  Each row is the power time course
of a single subject, at a single electrode, at a single wavelet frequency — and
PCA/ICA treats it as a single observation described by T features.

### Comparison with the super-brain approach

| | Super-brain (Approach 1) | Inverted super-brain (Approach 4) |
|---|---|---|
| **Observations** | Time (T) | Subject × Channel × Frequency (S×C×F) |
| **Features** | Subject × Channel × Frequency (S×C×F) | Time (T) |
| **Components are** | S×C×F loading patterns (spatial–spectral–subject) | Temporal patterns (time courses) |
| **Scores are** | Time courses (length T) | S×C×F weight vectors (decomposable into subject, channel, frequency) |

### What the decomposition finds

PCA / ICA applied to this matrix discover a small set of **temporal components**
(each of length T).  However, unlike the super-brain, here the components are the
*features* (time patterns) rather than the scores.  The score vector for each
component has S × C × F entries, which can be *reshaped back* to
`(n_subjects, n_channels, n_freqs)` and decomposed into three interpretable axes:

| Quantity | Shape | Interpretation |
|----------|-------|----------------|
| **Component pattern** | `(n_times,)` | A temporal pattern shared across subject–channel–frequency observations |
| **Score vector** | `(n_subjects × n_channels × n_freqs,)` | How strongly each observation loads on this temporal pattern |
| **Subject loadings** | `(n_subjects,)` | Mean absolute score over channels and frequencies — which participants contribute most |
| **Channel loadings** | `(n_channels,)` | Mean absolute score over subjects and frequencies — spatial topography |
| **Frequency loadings** | `(n_freqs,)` | Mean absolute score over subjects and channels — spectral profile |

### Interpretation guide

- A component whose **temporal pattern** shows a burst at a specific time
  (e.g. in response to a musical event), with **high subject loadings across
  all participants** and a **narrow frequency band**, would indicate a shared
  stimulus-locked response.
- Components with **consistent channel loadings** across subjects (high
  inter-individual correlation) reflect stimulus-driven spatial patterns.
- The key advantage of this approach is that **many observations** (S×C×F) are
  described by **few features** (T after subsampling), making ICA estimation
  statistically robust — the ratio of observations to features is very high.
- Each component's temporal pattern directly shows **when** the shared neural
  processing occurs, and the score decomposition shows **where** (channels),
  **at what frequency**, and **in whom**.

### Analyses

1. PCA scree plot (variance explained)
2. ICA temporal component patterns
3. Subject loadings per IC
4. ICA channel loadings — topographic maps (mean across subjects & frequencies)
5. ICA topographic maps — mean loading across subjects
6. ICA topographic maps — variance across subjects
7. Frequency loadings per IC
8. ICA component spectrograms (time–frequency view via STFT of component pattern)
9. Inter-subject correlation of IC channel loadings
10. Cross-component correlation matrix (PCA vs ICA)

## Configuration


In [ ]:
# ── Experiment configuration ─────────────────────────────────────────────────
CONDITION = ConditionVariants.PLACEBO
MUSIC_TYPES = [MusicTypeVariants.CLASSICAL]  # single type for fast exploration
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]
PROCESS_AND_SAVE_DATA = False  # set True to re-process raw files

# ── Wavelet settings ─────────────────────────────────────────────────────────
REPRESENTATION = "power"
WAVELET_FREQ_MIN = min(lo for lo, _ in FREQUENCY_BANDS.values())
WAVELET_FREQ_MAX = max(hi for _, hi in FREQUENCY_BANDS.values())
WAVELET_N_FREQS = max(
    2,
    int(round((WAVELET_FREQ_MAX - WAVELET_FREQ_MIN) / WAVELET_BAND_FREQ_RESOLUTION_HZ))
    + 1,
)
FREQS = np.linspace(WAVELET_FREQ_MIN, WAVELET_FREQ_MAX, WAVELET_N_FREQS)

KEEP_FREQUENCY_DIM = True
RESHAPE_FREQUENCY_DIM = True  # -> (n_subjects, n_channels, n_freqs, n_times)

# ── Reuse / compute ──────────────────────────────────────────────────────────
REUSE_WAVELETS = True  # load from cache; set False to compute + save

# ── Subject subset ────────────────────────────────────────────────────────────
N_SUBJECTS_SUBSET: int | None = 5

# ── Channel and time subset ───────────────────────────────────────────────────
N_CHANNELS_SUBSET: int | None = 32  # first N channels (from 195)
N_TIMES_SUBSET: int | None = 10000  # first N time samples

# ── Decomposition settings ────────────────────────────────────────────────────
N_COMPONENTS_PCA = 20  # number of PCA components to retain
N_COMPONENTS_ICA = 10  # number of ICA components to extract
ICA_RANDOM_STATE = 42  # reproducibility

# ── Storage directory ─────────────────────────────────────────────────────────
WAVELET_DIR: Path = (
    ProjectPaths.NOTEBOOKS_DIR / "04-wavelet-ica-analysis" / "wavelet_cache"
)

# ── Plot saving ──────────────────────────────────────────────────────────────
SAVE_PLOTS = True
# Canonical notebook plot output layout (mirrors the production CLI
# script scripts/run_wavelet_ica.py):
#   notebooks/04-wavelet-ica-analysis/plots/<approach>/pca_ica/
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "04-wavelet-ica-analysis"
    / "plots"
    / "inverted_superbrain"
    / "pca_ica"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Wavelet base directory : {WAVELET_DIR}")
print(
    f"Frequencies            : {FREQS[0]:.1f}\u2013{FREQS[-1]:.1f} Hz ({len(FREQS)} steps)"
)
print(f"PCA components         : {N_COMPONENTS_PCA}")
print(f"ICA components         : {N_COMPONENTS_ICA}")

## Data Loading


In [ ]:
analyzers = load_analyzers(
    MUSIC_TYPES,
    CONDITION,
    EXCLUSION_CATEGORIES,
    PROCESS_AND_SAVE_DATA,
    normalize_data=False,
)
datasets = analyzers_to_datasets(analyzers)

# Always limit to first N_SUBJECTS_SUBSET individuals
if N_SUBJECTS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:N_SUBJECTS_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_SUBJECTS_SUBSET} individuals.")

# Slice to channel subset
if N_CHANNELS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :N_CHANNELS_SUBSET, :])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_CHANNELS_SUBSET} channels.")

# Slice to time subset
if N_TIMES_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :, :N_TIMES_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_TIMES_SUBSET} time samples.")

print("Loaded datasets:", list(datasets.keys()))
for label, ad in datasets.items():
    print(
        f"  {label}: {ad.n_items} subjects, {ad.n_features} channels, "
        f"{ad.n_samples} samples"
    )

## Load or Compute Wavelet Transforms

Stored in `WAVELET_DIR/broadband/`.


In [ ]:
broadband_datasets = wavelet_transform(
    datasets=datasets,
    freqs=FREQS,
    representation=REPRESENTATION,
    keep_frequency_dim=KEEP_FREQUENCY_DIM,
    reshape_frequency_dim=RESHAPE_FREQUENCY_DIM,
    wavelet_dir=WAVELET_DIR / "broadband",
    reuse_wavelets=REUSE_WAVELETS,
)
for label, ad in broadband_datasets.items():
    source = ad.metadata.get("loaded_from_wavelet_file", "computed_now")
    print(f"[broadband] {label}: shape={ad.data.shape}  source={source}")

## Dataset Selection

Change `LABEL` to switch between music types.  The remaining cells use
`bb_data` (broadband wavelet power, 4-D) and derived quantities.


In [ ]:
LABEL = list(broadband_datasets.keys())[0]

bb_ad = broadband_datasets[LABEL]
bb_data = bb_ad.data  # (n_subjects, n_channels, n_freqs, n_times)
sfreq = bb_ad.sfreq

n_subjects, n_channels, n_freqs, n_times = bb_data.shape
time = np.arange(n_times) / sfreq

print(f"Dataset    : {LABEL}")
print(
    f"Shape      : {bb_data.shape}  (subjects \u00d7 channels \u00d7 freqs \u00d7 times)"
)
print(f"Duration   : {n_times / sfreq:.1f} s  @  {sfreq} Hz")
print(f"Freq range : {FREQS[0]:.1f}\u2013{FREQS[-1]:.1f} Hz ({n_freqs} steps)")

## Inverted Super-Brain Reshape

Flatten `(n_subjects, n_channels, n_freqs)` into one **observation** axis.
The resulting matrix has shape `(n_subjects × n_channels × n_freqs, n_times)`.

Unlike the super-brain (Approach 1), here rows are **observations** and columns
(time points) are **features**.  Before decomposition we **z-score along
the time axis** so that each `(subject, channel, frequency)` slice has
mean 0 and unit variance.


In [ ]:
# Z-score along time: each (subject, channel, frequency) slice → mean=0, std=1
bb_z = zscore_by_time(bb_data)  # (S, C, F, T)

# Reshape 4-D → 2-D:  (n_subjects * n_channels * n_freqs,  n_times)
X_z = bb_z.reshape(-1, n_times)  # (S*C*F, T)
print(f"Inverted super-brain matrix shape: {X_z.shape}")
print("Z-scored matrix ready.")
print(f"Observations (S×C×F): {X_z.shape[0]}")
print(f"Features     (T)    : {X_z.shape[1]}")

---
## 1 — PCA Scree Plot (Variance Explained)

Fit PCA to the inverted super-brain matrix.  Now observations are
subject–channel–frequency triplets and features are time points.
`pca.fit(X_z)` with sklearn expects `(n_samples, n_features)` =
`(S*C*F, T)`.


In [ ]:
pca = PCA(n_components=N_COMPONENTS_PCA, random_state=ICA_RANDOM_STATE)
pca.fit(X_z)  # sklearn expects (n_samples, n_features) → (S*C*F, T)

explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: individual variance
axes[0].bar(range(1, len(explained) + 1), explained, color="steelblue")
axes[0].set_xlabel("Component")
axes[0].set_ylabel("Variance explained")
axes[0].set_title(f"PCA Scree Plot — {LABEL}")

# Panel B: cumulative
axes[1].plot(range(1, len(cumulative) + 1), cumulative, "o-", color="coral")
axes[1].axhline(0.9, ls="--", color="gray", label="90%")
axes[1].set_xlabel("Number of components")
axes[1].set_ylabel("Cumulative variance explained")
axes[1].set_title(f"Cumulative Variance — {LABEL}")
axes[1].legend()

fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "pca_scree.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

print(
    f"Top {N_COMPONENTS_PCA} components explain "
    f"{cumulative[-1] * 100:.1f}% of total variance."
)

---
## 2 — ICA Temporal Component Patterns

Fit FastICA and extract the **component vectors** — each is a temporal
pattern of length T.  These represent the time-domain features that
best separate the observations (subject–channel–frequency triplets)
into independent groups.


In [ ]:
ica = FastICA(
    n_components=N_COMPONENTS_ICA,
    random_state=ICA_RANDOM_STATE,
    max_iter=500,
    whiten="unit-variance",
)
ica_scores = ica.fit_transform(X_z)  # (S*C*F, K_ica)

# ICA components: (K_ica, T) — each row is a temporal pattern
ica_components = ica.components_  # (K_ica, T)

n_show = min(6, N_COMPONENTS_ICA)
_ylim = np.percentile(np.abs(ica_components[:n_show]), 99)
fig, axes = plt.subplots(n_show, 1, figsize=(14, 2.5 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    ax.plot(time, ica_components[i], lw=0.8, color="darkorange")
    ax.set_ylim(-_ylim, _ylim)
    ax.set_ylabel(f"IC {i + 1}")
    ax.set_title(f"ICA Temporal Pattern {i + 1}", fontsize=10)

axes[-1].set_xlabel("Time (s)")
fig.suptitle(f"ICA Temporal Component Patterns — {LABEL}", fontsize=13, y=1.01)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_temporal_patterns.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## 3 — Subject Loadings per IC

The ICA scores have shape `(S*C*F, K_ica)`.  Reshape to
`(S, C, F, K)` and average absolute values over channels and frequencies
to obtain a **per-subject weight** for each component.

This shows which participants contribute most strongly to each IC.


In [ ]:
# ICA scores: (S*C*F, K_ica)  →  reshape to (S, C, F, K)
scores_4d = ica_scores.reshape(n_subjects, n_channels, n_freqs, N_COMPONENTS_ICA)

# Subject loadings: mean |score| over channels and frequencies
subject_loadings = np.abs(scores_4d).mean(axis=(1, 2))  # (S, K)

n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 4), sharey=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    ax.barh(
        range(n_subjects),
        subject_loadings[:, i],
        color="darkorange",
    )
    ax.set_yticks(range(n_subjects))
    ax.set_yticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=8)
    ax.set_xlabel("|score|")
    ax.set_title(f"IC {i + 1}", fontsize=10)

axes[0].set_ylabel("Subject")
fig.suptitle(f"Subject Loadings (ICA) — {LABEL}", fontsize=13, y=1.02)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_subject_loadings.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## 4 — ICA Channel Loadings (Topographic Maps)

Average absolute ICA scores over subjects and frequencies to get a
per-channel weight for each IC.  Plot as scalp topographies using
`mne.viz.plot_topomap`.


In [ ]:
# Channel loadings: mean |score| over subjects and frequencies
channel_loadings = np.abs(scores_4d).mean(axis=(0, 2))  # (C, K)

# Get MNE Info for topomap
info = analyzers[LABEL].info
info = mne.pick_info(info, mne.pick_types(info, eeg=True))
if n_channels < len(info.ch_names):
    info = mne.pick_info(info, list(range(n_channels)))

n_show = min(6, N_COMPONENTS_ICA)
_vlim = np.percentile(np.abs(channel_loadings[:, :n_show]), 99)
fig, axes = plt.subplots(1, n_show, figsize=(3.5 * n_show, 4))
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    im, _ = plot_topomap(
        channel_loadings[:, i],
        info,
        axes=ax,
        show=False,
        cmap="RdBu_r",
    )
    ax.set_title(f"IC {i + 1}", fontsize=10)

fig.suptitle(f"ICA Channel Loadings (mean |score|) — {LABEL}", fontsize=12)
plt.colorbar(im, ax=axes[-1], label="mean |score|")
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_channel_topomap.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## 5 — ICA Topographic Maps — Mean Loading Across Subjects

The ICA scores reshaped to `(S, C, F, K)` give per-(subject, channel,
frequency) weights.  Average over frequencies to get per-(subject, channel)
loadings, then average over subjects for the **mean channel loading**
per IC.  Plot as scalp topographies.


In [ ]:
# Per-subject channel loadings: average scores over frequencies → (S, C, K)
ica_channel_loadings = scores_4d.mean(axis=2)  # (S, C, K)

# Mean across subjects → (C, K)
ica_ch_mean = ica_channel_loadings.mean(axis=0)  # (C, K)

n_show = min(6, N_COMPONENTS_ICA)
_vlim_mean = np.percentile(np.abs(ica_ch_mean[:, :n_show]), 99)
fig, axes = plt.subplots(1, n_show, figsize=(3.5 * n_show, 4))
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    im, _ = plot_topomap(
        ica_ch_mean[:, i],
        info,
        axes=ax,
        show=False,
        cmap="RdBu_r",
        vlim=(-_vlim_mean, _vlim_mean),
    )
    ax.set_title(f"IC {i + 1}", fontsize=10)

fig.suptitle(
    f"ICA Topomaps — Mean Loading Across Subjects — {LABEL}",
    fontsize=12,
)
plt.colorbar(im, ax=axes[-1], label="mean loading")
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_topomap_mean.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## 6 — ICA Topographic Maps — Variance Across Subjects

Compute the **variance** of each channel's ICA loading across subjects.
High variance indicates the IC has different strength at that electrode
across individuals.  Low variance means consistent activation.


In [ ]:
# Variance across subjects → (C, K)
ica_ch_var = ica_channel_loadings.var(axis=0)  # (C, K)

n_show = min(6, N_COMPONENTS_ICA)
_vmax_var = np.percentile(ica_ch_var[:, :n_show], 99)
fig, axes = plt.subplots(1, n_show, figsize=(3.5 * n_show, 4))
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    im, _ = plot_topomap(
        ica_ch_var[:, i],
        info,
        axes=ax,
        show=False,
        cmap="YlOrRd",
        vlim=(0, _vmax_var),
    )
    ax.set_title(f"IC {i + 1}", fontsize=10)

fig.suptitle(
    f"ICA Topomaps — Variance Across Subjects — {LABEL}",
    fontsize=12,
)
plt.colorbar(im, ax=axes[-1], label="variance of loading")
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_topomap_variance.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## 7 — Frequency Loadings per IC

Average absolute ICA scores over subjects and channels to get a
per-frequency weight for each IC.  This reveals which frequency
bands dominate each component.


In [ ]:
# Frequency loadings: mean |score| over subjects and channels
freq_loadings = np.abs(scores_4d).mean(axis=(0, 1))  # (F, K)

n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(n_show, 1, figsize=(10, 2.5 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

band_colors = {
    "delta": "#d4e6f1",
    "theta": "#d5f5e3",
    "alpha": "#fdebd0",
    "beta": "#fadbd8",
    "gamma": "#e8daef",
}

_ymax = np.percentile(freq_loadings[:, :n_show], 99)
for i, ax in enumerate(axes):
    ax.plot(FREQS, freq_loadings[:, i], lw=1.5, color="darkorange")
    ax.set_ylim(0, _ymax)
    for band, (lo, hi) in FREQUENCY_BANDS.items():
        ax.axvspan(lo, hi, alpha=0.2, color=band_colors.get(band, "grey"))
    ax.set_ylabel("|score|")
    ax.set_title(f"IC {i + 1}", fontsize=10)

axes[-1].set_xlabel("Frequency (Hz)")
fig.suptitle(f"Frequency Loadings (ICA) — {LABEL}", fontsize=13, y=1.01)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_freq_loadings.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## 8 — ICA Component Spectrograms

For each ICA temporal component pattern, compute the short-time Fourier
transform (via Welch's method in sliding windows) to reveal its
time–frequency content.  This shows whether a component has a stable
spectral signature or varies over the stimulus.


In [ ]:
n_show = min(4, N_COMPONENTS_ICA)
fig, axes = plt.subplots(n_show, 1, figsize=(14, 3 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    f_spec, t_spec, Sxx = sp_spectrogram(
        ica_components[i], fs=sfreq, nperseg=int(sfreq * 2), noverlap=int(sfreq)
    )
    # Limit to the wavelet frequency range
    freq_mask = f_spec <= FREQS[-1]
    _Sxx_db = 10 * np.log10(Sxx[freq_mask] + 1e-12)
    _vmin_s, _vmax_s = np.percentile(_Sxx_db, 1), np.percentile(_Sxx_db, 99)
    ax.pcolormesh(
        t_spec, f_spec[freq_mask], _Sxx_db, cmap="inferno", vmin=_vmin_s, vmax=_vmax_s
    )
    ax.set_ylabel("Freq (Hz)")
    ax.set_title(f"IC {i + 1} spectrogram", fontsize=10)

axes[-1].set_xlabel("Time (s)")
fig.suptitle(f"ICA Component Spectrograms — {LABEL}", fontsize=13, y=1.01)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_spectrograms.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## 9 — Inter-Individual IC Correlations

For each ICA component, compute the Pearson correlation between every
pair of subjects using their channel loading vectors (freq-averaged).
High correlations indicate the IC has a consistent spatial distribution
across individuals — a hallmark of stimulus-driven components.


In [ ]:
# ica_channel_loadings: (S, C, K) — per-subject, per-channel IC loading
n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(
    1, n_show, figsize=(3.5 * n_show, 3.5), constrained_layout=True
)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    # Each subject's channel-loading vector for IC i: (n_channels,)
    corr_mat = np.corrcoef(ica_channel_loadings[:, :, i])  # (S, S)
    im = ax.imshow(corr_mat, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(n_subjects))
    ax.set_yticks(range(n_subjects))
    ax.set_xticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=7)
    ax.set_yticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=7)
    ax.set_title(f"IC {i + 1}", fontsize=10)

fig.suptitle(
    f"Inter-Individual IC Correlations (channel loadings) — {LABEL}",
    fontsize=12,
)
plt.colorbar(im, ax=axes[-1], label="Pearson r", shrink=0.8)
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "ica_interindividual_correlation.png",
        dpi=150,
        bbox_inches="tight",
    )
plt.show()
plt.close("all")

---
## 10 — Cross-Component Correlation Matrix (PCA vs ICA)

PCA components (scores) are orthogonal by construction; ICA components may
have residual correlations.  We compare the Pearson correlation matrices of
PCA and ICA scores side by side.


In [ ]:
# PCA scores: (S*C*F, K_pca)
pca_scores = pca.transform(X_z)  # (S*C*F, K_pca)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel A: PCA correlation
pca_corr = np.corrcoef(pca_scores.T)
im0 = axes[0].imshow(pca_corr, vmin=-1, vmax=1, cmap="RdBu_r")
axes[0].set_title("PCA score correlations")
axes[0].set_xlabel("Component")
axes[0].set_ylabel("Component")
plt.colorbar(im0, ax=axes[0], shrink=0.8)

# Panel B: ICA correlation
ica_corr = np.corrcoef(ica_scores.T)
im1 = axes[1].imshow(ica_corr, vmin=-1, vmax=1, cmap="RdBu_r")
axes[1].set_title("ICA score correlations")
axes[1].set_xlabel("Component")
axes[1].set_ylabel("Component")
plt.colorbar(im1, ax=axes[1], shrink=0.8)

fig.suptitle(f"Cross-Component Correlations — {LABEL}", fontsize=13, y=1.01)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "cross_component_correlation.png", dpi=150, bbox_inches="tight"
    )
plt.show()
plt.close("all")

---
## Summary

Available variables for further analysis:

| Variable | Shape | Description |
|----------|-------|-------------|
| `X_z` | `(S×C×F, T)` | Z-scored inverted super-brain matrix |
| `pca` | — | Fitted PCA object |
| `pca_scores` | `(S×C×F, K)` | PCA scores (observation weights) |
| `ica` | — | Fitted FastICA object |
| `ica_scores` | `(S×C×F, K_ica)` | ICA scores (observation weights) |
| `ica_components` | `(K_ica, T)` | ICA temporal component patterns |
| `scores_4d` | `(S, C, F, K)` | ICA scores reshaped to 4-D |
| `ica_channel_loadings` | `(S, C, K)` | Per-subject channel loadings (freq-averaged ICA scores) |

**Analyses implemented:**

1. PCA scree plot (variance explained)
2. ICA temporal component patterns
3. Subject loadings per IC
4. ICA channel loadings — topographic maps
5. ICA topomaps — mean loading across subjects
6. ICA topomaps — variance across subjects
7. Frequency loadings per IC
8. ICA component spectrograms (STFT of component patterns)
9. Inter-individual IC correlations (channel loading vectors)
10. Cross-component correlation (PCA vs ICA)

See `README.md` in this directory for the full analysis rationale,
alternative decomposition strategies, and ideas for future extensions.